# EduBénin — Fine-tuning GLM-4.7
# Exécuter sur Google Colab avec GPU T4

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
vram = torch.cuda.get_device_properties(0).total_memory
print(f"VRAM: {vram/1e9:.1f} GB")

In [ ]:
!pip install transformers datasets peft trl \
  accelerate bitsandbytes sentencepiece \
  fastapi uvicorn nest-asyncio pyngrok

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/MyDrive/EduBenin'
print(f"Drive monté: {DRIVE_PATH}")

In [ ]:
import json
dataset_path = f'{DRIVE_PATH}/dataset/edubenin_dataset_v2.json'
with open(dataset_path) as f:
    data = json.load(f)
print(f"Total exemples: {len(data)}")

In [ ]:
def formater_glm(exemple):
    return {
        "conversations": [
            {
                "role": "system",
                "content": "Tu es EduBot, tuteur IA du système éducatif secondaire béninois. Tu réponds en français avec des exemples béninois."
            },
            {
                "role": "user",
                "content": exemple["instruction"]
            },
            {
                "role": "assistant",
                "content": exemple["output"]
            }
        ]
    }
dataset_glm = [formater_glm(ex) for ex in data]
print(f"Dataset formaté: {len(dataset_glm)} exemples")

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)
import torch

MODEL_NAME = "THUDM/glm-4-9b-chat"
print("Chargement tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)
print("Chargement modèle (peut prendre 10min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    load_in_4bit=True
)
print("✅ Modèle chargé!")

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

train_ds = Dataset.from_list(dataset_glm)
output_dir = f'{DRIVE_PATH}/model/edubot_glm47'
args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
)
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    max_seq_length=2048,
)
print("🚀 Début du fine-tuning...")
trainer.train()
print("✅ Fine-tuning terminé!")

In [ ]:
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"✅ Modèle sauvegardé: {output_dir}")

In [ ]:
prompt = """Tu es EduBot.
Explique la loi d'Ohm à un élève de 3ème au Bénin."""
inputs = tokenizer(
    prompt, return_tensors="pt"
).to("cuda")
output = model.generate(
    **inputs, max_new_tokens=300
)
print(tokenizer.decode(output[0]))

In [ ]:
# Lance le même serveur FastAPI
# mais avec GLM-4.7 local comme backend
# Expose via ngrok pour le backend Node.js

import subprocess, os
!pip install pyngrok
from pyngrok import ngrok
import nest_asyncio

NGROK_TOKEN = ""  # Mets ton token ngrok gratuit ici
ngrok.set_auth_token(NGROK_TOKEN)

# Clone le repo et lance FastAPI
!git clone https://github.com/EduBenin/edubenin-ia.git
os.chdir("edubenin-ia")

# Configure pour mode colab
with open(".env", "w") as f:
    f.write("MODEL_MODE=openai\n")
    f.write(f"OPENAI_API_KEY=\n")

!pip install -r requirements.txt

import uvicorn
nest_asyncio.apply()

public_url = ngrok.connect(8000)
print(f"🌍 URL EduBot publique: {public_url}")
print("Copie cette URL dans ton .env backend:")
print(f"IA_SERVICE_URL={public_url}")

uvicorn.run(
    "app.main:app",
    host="0.0.0.0",
    port=8000
)